# FRAGO 4 — La Frappe d'Artillerie

## Contexte opérationnel

Vous êtes analyste OSINT pour l'Opération Manticore. 
Un commandement allié vous demande d'identifier **la cible infrastructurelle la plus critique**
dans la zone d'opération, définie comme l'ouvrage d'art le plus connecté du réseau.

Votre livrable : les **coordonnées GPS précises** de la cible, avec une carte de situation.

## Objectifs pédagogiques

1. Utiliser Neo4j pour identifier un nœud par **centralité de degré**
2. Croiser avec PostGIS pour récupérer les **coordonnées géographiques réelles**
3. Produire une **visualisation cartographique** avec geopandas/matplotlib
4. Présenter le résultat en 2 minutes (format briefing militaire)


In [ ]:
# Imports
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import psycopg2
from neo4j import GraphDatabase
from dotenv import load_dotenv
from shapely import wkt

load_dotenv(dotenv_path='../.env')
print('[OK] Imports OK')

In [ ]:
# Connexions aux bases de données

def get_neo4j():
    uri  = os.getenv('NEO4J_URI',      'bolt://localhost:7687')
    user = os.getenv('NEO4J_USER',     'neo4j')
    pwd  = os.getenv('NEO4J_PASSWORD', 'manticore2026')
    driver = GraphDatabase.driver(uri, auth=(user, pwd))
    driver.verify_connectivity()
    print(f'[OK] Neo4j connecté : {uri}')
    return driver

def get_postgis():
    conn = psycopg2.connect(
        host=os.getenv('POSTGIS_HOST', 'localhost'),
        port=os.getenv('POSTGIS_PORT', 5432),
        dbname=os.getenv('POSTGIS_DB', 'bdtopo_manticore'),
        user=os.getenv('POSTGIS_USER', 'postgres'),
        password=os.getenv('POSTGIS_PASSWORD', 'manticore2026'),
    )
    print('[OK] PostGIS connecté')
    return conn

driver = get_neo4j()
conn   = get_postgis()

## Étape 1 — Identification de la cible (Neo4j)

On cherche le pont avec la **centralité de degré la plus élevée** :
c'est-à-dire celui qui est impliqué dans le plus grand nombre de relations
dans le graphe d'ontologie.

En pratique, un nœud très connecté dans l'ontologie correspond à un type d'objet
qui apparaît dans beaucoup de contextes différents — c'est un point critique.


In [ ]:
# Étape 1 : Centralité de degré dans Neo4j
# Trouver le pont (ou ouvrage) le plus connecté du graphe

cypher = """
    MATCH (n:ClasseOntologie)
    WHERE n.name CONTAINS 'Pont'
       OR n.name CONTAINS 'ouvrage'
       OR n.sql_name = 'construction_lineaire'
    OPTIONAL MATCH (n)-[r]-()
    RETURN
        n.name     AS nom,
        n.sql_name AS table_postgis,
        n.obj_type AS type,
        count(r)   AS degre
    ORDER BY degre DESC
    LIMIT 10
"""

with driver.session() as session:
    result  = session.run(cypher)
    records = result.data()

df_cibles = pd.DataFrame(records)
print('Top 10 nœuds par centralité de degré :')
display(df_cibles)

# La cible principale
cible = df_cibles.iloc[0]
print(f"\n>>> CIBLE DÉSIGNÉE : {cible['nom']} (degré={cible['degre']}, table={cible['table_postgis']})")

## Étape 2 — Coordonnées GPS (PostGIS)

Maintenant qu'on a identifié le **type** de la cible via Neo4j,
on récupère les instances réelles (avec leurs géométries) dans PostGIS.

On cherche les ponts de ce type dans la zone d'opération,
et on sélectionne celui avec la géométrie la plus représentative (centroïde).


In [ ]:
# Étape 2 : Récupérer les coordonnées GPS depuis PostGIS

# TODO : remplacez '???' par le cleabs identifié à l'étape précédente si vous en avez un spécifique
# Sinon, on cherche tous les ponts de la table identifiée

table_cible = cible['table_postgis'] or 'construction_lineaire'

query = f"""
    SELECT
        cleabs,
        nature,
        ST_AsText(ST_Centroid(geometrie)) AS coords,
        ST_AsText(geometrie)              AS geom_wkt
    FROM {table_cible}
    WHERE nature ILIKE 'Pont%'
    ORDER BY ST_Length(geometrie) DESC
    LIMIT 20;
"""

try:
    df_ponts = pd.read_sql(query, conn)
    print(f'{len(df_ponts)} ponts trouvés dans {table_cible}')
    display(df_ponts[['cleabs', 'nature', 'coords']].head(10))

    # Sélectionner la cible finale (le plus long pont = le plus stratégique)
    cible_finale = df_ponts.iloc[0]
    print(f"\n>>> COORDONNÉES DE LA CIBLE : {cible_finale['coords']}")
    print(f"    cleabs : {cible_finale['cleabs']}")
    print(f"    nature : {cible_finale['nature']}")
except Exception as e:
    print(f'[ERREUR] {e}')
    print("→ Vérifiez que le dump SQL a été chargé et que la table existe.")
    df_ponts = pd.DataFrame()

## Étape 3 — Livrable

### Format du briefing (2 minutes)

Votre présentation doit inclure :

1. **SITUATION** : Quel est l'objet ciblé ? Quel est son identifiant (cleabs) ?
2. **MÉTHODE** : Comment avez-vous identifié cette cible ? (centralité de degré Neo4j)
3. **COORDONNÉES** : Quelles sont les coordonnées GPS précises ?
4. **IMPACT** : Pourquoi ce pont est-il stratégique ? (connexions, trafic estimé, alternatives)
5. **CARTE** : Montrez la carte de situation (cellule suivante)

### Critères d'évaluation

- [ ] La cible est identifiée dans Neo4j par centralité de degré
- [ ] Les coordonnées GPS sont récupérées depuis PostGIS
- [ ] La carte affiche la cible et son contexte géographique
- [ ] Le briefing est clair, structuré, en 2 minutes chrono


In [ ]:
# Étape 3 : Visualisation cartographique de la cible

if df_ponts.empty:
    print("[SKIP] Pas de données à afficher — chargez d'abord le dump SQL.")
else:
    # Convertir les géométries WKT en GeoDataFrame
    df_ponts['geometry'] = df_ponts['geom_wkt'].apply(
        lambda wkt_str: wkt.loads(wkt_str) if wkt_str else None
    )
    gdf = gpd.GeoDataFrame(df_ponts, geometry='geometry', crs='EPSG:4326')

    # Préparer la cible principale
    cible_geom = wkt.loads(cible_finale['coords'])  # centroïde
    gdf_cible = gpd.GeoDataFrame(
        {'label': [cible_finale['cleabs']]},
        geometry=[cible_geom],
        crs='EPSG:4326'
    )

    # Figure
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.set_facecolor('#1a1a2e')  # fond sombre (style opérationnel)
    fig.patch.set_facecolor('#0f0f1a')

    # Afficher tous les ponts
    gdf.plot(
        ax=ax,
        color='#4fc3f7',
        linewidth=1.5,
        alpha=0.6,
        label='Ponts (zone d\'opération)'
    )

    # Marquer la cible
    gdf_cible.plot(
        ax=ax,
        color='red',
        markersize=200,
        marker='X',
        zorder=10,
        label='CIBLE'
    )

    # Annotation
    ax.annotate(
        f"CIBLE\n{cible_finale['cleabs'][:12]}...",
        xy=(cible_geom.x, cible_geom.y),
        xytext=(10, 10),
        textcoords='offset points',
        color='red',
        fontsize=9,
        fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#1a1a2e', edgecolor='red'),
    )

    ax.set_title(
        'OPÉRATION MANTICORE — Carte de situation\nFRAGO 4 : Frappe d\'artillerie',
        color='white', fontsize=14, fontweight='bold', pad=15
    )
    ax.tick_params(colors='#aaaaaa')
    ax.set_xlabel('Longitude', color='#aaaaaa')
    ax.set_ylabel('Latitude',  color='#aaaaaa')
    ax.spines[:].set_color('#333355')

    # Légende
    legend = ax.legend(
        facecolor='#1a1a2e',
        edgecolor='#4fc3f7',
        labelcolor='white',
        fontsize=9
    )

    plt.tight_layout()
    plt.savefig('../data/carte_cible.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("[OK] Carte sauvegardée dans data/carte_cible.png")